In [1]:
import os
import pandas as pd
import re
from textdistance import damerau_levenshtein

tables_folder = "./tables"
results_folder_1 = "./results_1"
results_folder_2 = "./results_2"
results_folder_3 = "./results_3"

group_by_size = "size"
group_by_font = "font"
group_by_borders = "borders"
group_by_scale = "scale"

method_title_1 = "1 способ: стандартная связка img2table + Tesseract без дополнительных настроек"
method_title_2 = "2 способ: разметка img2table + отдельные вызовы Tesseract для каждой размеченной ячейки"
method_title_3 = "3 способ: разметка img2table + один вызов Tesseract и распределение текста по ячейкам"

In [2]:
def pad_to_same_size(df1, df2):
    max_rows = max(df1.shape[0], df2.shape[0])
    max_cols = max(df1.shape[1], df2.shape[1])
    
    df1_padded = pd.DataFrame("", index=range(max_rows), columns=range(max_cols))
    df2_padded = pd.DataFrame("", index=range(max_rows), columns=range(max_cols))
    
    df1_padded.iloc[:df1.shape[0], :df1.shape[1]] = df1.values
    df2_padded.iloc[:df2.shape[0], :df2.shape[1]] = df2.values
    
    return df1_padded, df2_padded

def evaluate(table_true_path, table_pred_path):
    df_true = pd.read_csv(table_true_path, sep=";", dtype=str, encoding="utf-8-sig").fillna("")
    df_pred = pd.read_excel(table_pred_path, header=None, dtype=str).fillna("")
    
    df_true, df_pred = pad_to_same_size(df_true, df_pred)
    
    total_true_chars = 0
    total_distance = 0
    
    for i in range(df_true.shape[0]):
        for j in range(df_true.shape[1]):
            s_true = df_true.iloc[i, j]
            s_pred = df_pred.iloc[i, j]
            
            s_true = re.sub(r"\s+", "", s_true)
            s_pred = re.sub(r"\s+", "", s_pred)
            
            total_true_chars += len(s_true)
            total_distance += damerau_levenshtein(s_true, s_pred)
    
    accuracy = 1 - total_distance / total_true_chars
    
    return total_distance, accuracy

def get_results(tables_folder, results_folder):
    results = []
    for filename in os.listdir(results_folder):
        if not filename.endswith(".xlsx"):
            continue

        filename_match = re.search(r"(table_\d+)", filename)
        if not filename_match:
            continue
        table_true_name = filename_match.group(1)

        distance, accuracy = evaluate(
            table_true_path=os.path.join(tables_folder, f"{table_true_name}.csv"),
            table_pred_path=os.path.join(results_folder, filename),
        )

        filename_parts = filename.replace(".xlsx", "").split("_")
        size = int(filename_parts[1])
        font = filename_parts[2]
        borders = "contrast" in filename_parts
        scale = int(filename_parts[-1])

        results.append({
            "size": size,
            "font": font,
            "borders": borders,
            "scale": scale,
            "distance": distance,
            "accuracy": accuracy
        })

    if not results:
        return pd.DataFrame()
    return pd.DataFrame(results)

def agg_results(results, method_title, group_by="scale", verbose=0):
    if group_by not in ["size", "font", "borders", "scale"]:
        return pd.DataFrame()
    
    df = results.copy()
    
    if group_by == "size":
        group_column_name = "Размер"
        columns_order = ["3x3", "6x3", "6x5", "10x5", "3x3 (по центру)"]
        df[group_by] = df[group_by].map({
            1: columns_order[0], 2: columns_order[1], 3: columns_order[2],
            4: columns_order[3], 5: columns_order[4],
        })

    elif group_by == "font":
        group_column_name = "Шрифт"
        columns_order = ["Calibri", "GOST2304 Type B"]
        df[group_by] = df[group_by].map({"base": columns_order[0], "gost": columns_order[1]})

    elif group_by == "borders":
        group_column_name = "Рамки"
        columns_order = ["Светлые", "Контрастные"]
        df[group_by] = df[group_by].map({False: columns_order[0], True: columns_order[1]})

    else:
        group_column_name = "Масштаб"
        columns_order = sorted(df[group_by].unique(), reverse=True)

    df_agg = df.groupby(group_by).agg(
        count=("distance", "count"),
        max_distance=("distance", "max"),
        mean_distance=("distance", "mean"),
        min_distance=("distance", "min"),
        min_accuracy=("accuracy", "min"),
        mean_accuracy=("accuracy", "mean"),
        max_accuracy=("accuracy", "max"),
    ).reset_index()

    float_columns = ["mean_distance", "mean_accuracy", "min_accuracy", "max_accuracy"]
    df_agg[float_columns] = df_agg[float_columns].round(3)

    df_agg = df_agg.set_index(group_by).reindex(columns_order).reset_index()

    if group_by == "scale":
        df_agg[group_by] = df_agg[group_by].astype(str) + "%"

    df_agg.columns = [
        group_column_name,
        "Кол-во таблиц",
        "Макс. ошибок",
        "Средн. ошибок",
        "Мин. ошибок",
        "Мин. точность",
        "Средн. точность",
        "Макс. точность",
    ]

    if verbose:
        print(method_title)
        display(df_agg)

    return df_agg

In [3]:
(results_1 := get_results(tables_folder, results_folder_1))

,size,font,borders,scale,distance,accuracy
0,1,base,False,70,10,0.943820
1,1,base,False,100,4,0.977528
2,1,base,False,130,4,0.977528
3,1,base,False,160,0,1.000000
4,1,base,True,70,66,0.629213
...,...,...,...,...,...,...
67,5,gost,False,100,23,0.870787
68,5,gost,False,130,37,0.792135
69,5,gost,True,70,69,0.612360
70,5,gost,True,100,21,0.882022


In [4]:
(results_2 := get_results(tables_folder, results_folder_2))

,size,font,borders,scale,distance,accuracy
0,1,base,False,70,6,0.966292
1,1,base,False,100,0,1.000000
2,1,base,False,130,0,1.000000
3,1,base,False,160,0,1.000000
4,1,base,True,70,25,0.859551
...,...,...,...,...,...,...
67,5,gost,False,100,9,0.949438
68,5,gost,False,130,15,0.915730
69,5,gost,True,70,110,0.382022
70,5,gost,True,100,9,0.949438


In [5]:
(results_3 := get_results(tables_folder, results_folder_3))

,size,font,borders,scale,distance,accuracy
0,1,base,False,70,7,0.960674
1,1,base,False,100,0,1.000000
2,1,base,False,130,0,1.000000
3,1,base,False,160,0,1.000000
4,1,base,True,70,22,0.876404
...,...,...,...,...,...,...
67,5,gost,False,100,21,0.882022
68,5,gost,False,130,4,0.977528
69,5,gost,True,70,32,0.820225
70,5,gost,True,100,33,0.814607


In [6]:
print("Результаты распознавания таблиц, сгруппированные по количеству строк и столбцов:\n")
agg_results(results_1, method_title_1, group_by=group_by_size, verbose=1)
agg_results(results_2, method_title_2, group_by=group_by_size, verbose=1)
agg_results(results_3, method_title_3, group_by=group_by_size, verbose=1);

Результаты распознавания таблиц, сгруппированные по количеству строк и столбцов:

1 способ: стандартная связка img2table + Tesseract без дополнительных настроек


,Размер,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,3x3,16,69,27.688,0,0.612,0.844,1.000
1,6x3,16,200,69.688,3,0.524,0.834,0.993
2,6x5,16,317,96.812,0,0.386,0.812,1.000
3,10x5,12,565,191.083,6,0.381,0.791,0.993
4,3x3 (по центру),12,69,24.000,0,0.612,0.865,1.000


2 способ: разметка img2table + отдельные вызовы Tesseract для каждой размеченной ячейки


,Размер,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,3x3,16,104,16.062,0,0.416,0.910,1.000
1,6x3,16,158,30.562,2,0.624,0.927,0.995
2,6x5,16,223,58.875,5,0.568,0.886,0.990
3,10x5,12,353,98.000,9,0.613,0.893,0.990
4,3x3 (по центру),12,110,17.417,0,0.382,0.902,1.000


3 способ: разметка img2table + один вызов Tesseract и распределение текста по ячейкам


,Размер,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,3x3,16,55,13.625,0,0.691,0.923,1.000
1,6x3,16,128,32.938,1,0.695,0.922,0.998
2,6x5,16,165,49.688,1,0.680,0.904,0.998
3,10x5,12,299,86.000,2,0.673,0.906,0.998
4,3x3 (по центру),12,60,16.667,0,0.663,0.906,1.000


In [7]:
print("Результаты распознавания таблиц, сгруппированные по шрифту:\n")
agg_results(results_1, method_title_1, group_by=group_by_font, verbose=1)
agg_results(results_2, method_title_2, group_by=group_by_font, verbose=1)
agg_results(results_3, method_title_3, group_by=group_by_font, verbose=1);

Результаты распознавания таблиц, сгруппированные по шрифту:

1 способ: стандартная связка img2table + Tesseract без дополнительных настроек


,Шрифт,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,Calibri,36,285,28.583,0,0.591,0.940,1.000
1,GOST2304 Type B,36,565,129.417,21,0.381,0.719,0.882


2 способ: разметка img2table + отдельные вызовы Tesseract для каждой размеченной ячейки


,Шрифт,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,Calibri,36,153,18.111,0,0.779,0.964,1.000
1,GOST2304 Type B,36,353,67.250,9,0.382,0.844,0.952


3 способ: разметка img2table + один вызов Tesseract и распределение текста по ячейкам


,Шрифт,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,Calibri,36,100,11.167,0,0.876,0.976,1.000
1,GOST2304 Type B,36,299,65.833,4,0.663,0.850,0.978


In [8]:
print("Результаты распознавания таблиц, сгруппированные по типу рамок:\n")
agg_results(results_1, method_title_1, group_by=group_by_borders, verbose=1)
agg_results(results_2, method_title_2, group_by=group_by_borders, verbose=1)
agg_results(results_3, method_title_3, group_by=group_by_borders, verbose=1);

Результаты распознавания таблиц, сгруппированные по типу рамок:

1 способ: стандартная связка img2table + Tesseract без дополнительных настроек


,Рамки,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,Светлые,36,435,66.75,0,0.524,0.859,1.0
1,Контрастные,36,565,91.25,0,0.381,0.800,1.0


2 способ: разметка img2table + отдельные вызовы Tesseract для каждой размеченной ячейки


,Рамки,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,Светлые,36,218,30.694,0,0.672,0.932,1.0
1,Контрастные,36,353,54.667,0,0.382,0.876,1.0


3 способ: разметка img2table + один вызов Tesseract и распределение текста по ячейкам


,Рамки,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,Светлые,36,299,37.306,0,0.663,0.917,1.0
1,Контрастные,36,226,39.694,0,0.694,0.908,1.0


In [9]:
print("Результаты распознавания таблиц, сгруппированные по масштабу:\n")
agg_results(results_1, method_title_1, group_by=group_by_scale, verbose=1)
agg_results(results_2, method_title_2, group_by=group_by_scale, verbose=1)
agg_results(results_3, method_title_3, group_by=group_by_scale, verbose=1);

Результаты распознавания таблиц, сгруппированные по масштабу:

1 способ: стандартная связка img2table + Tesseract без дополнительных настроек


,Масштаб,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,160%,12,136,44.667,0,0.707,0.883,1.000
1,130%,20,186,50.050,0,0.663,0.881,1.000
2,100%,20,255,52.750,0,0.721,0.890,1.000
3,70%,20,565,154.800,10,0.381,0.685,0.944


2 способ: разметка img2table + отдельные вызовы Tesseract для каждой размеченной ячейки


,Масштаб,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,160%,12,53,16.667,0,0.882,0.956,1.000
1,130%,20,97,24.800,0,0.849,0.951,1.000
2,100%,20,94,20.650,0,0.880,0.960,1.000
3,70%,20,353,98.200,6,0.382,0.771,0.966


3 способ: разметка img2table + один вызов Tesseract и распределение текста по ячейкам


,Масштаб,Кол-во таблиц,Макс. ошибок,Средн. ошибок,Мин. ошибок,Мин. точность,Средн. точность,Макс. точность
0,160%,12,61,16.333,0,0.882,0.962,1.000
1,130%,20,96,24.100,0,0.843,0.948,1.000
2,100%,20,78,25.400,0,0.814,0.935,1.000
3,70%,20,299,79.300,6,0.663,0.826,0.966
